# Merge

Consolida los outputs de los notebooks distribuidos.

```
Bloque 1 — después de scraper
Bloque 2 — después de kb
Bloque 3 — después de sentiment + framer + argument + factual_backfill
```

In [ ]:
!git clone https://github.com/average-peruvian/media-framing.git
!mv media-framing/* .
!rm -rf media-framing
!pip install -e . -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT = '/content/drive/MyDrive/ant/selection/out'

import pandas as pd
from medianalysis.distrib           import merge_workers
from medianalysis.rhetoric.argument import explode_arguments, link_arguments_to_kb

### Bloque 1 — después de scraper

In [ ]:
merge_workers(f'{DRIVE_OUT}/noticias_cuerpo.csv', 'id', DRIVE_OUT)
merge_workers(f'{DRIVE_OUT}/clasificacion.csv',   'id', DRIVE_OUT)

# Filtrar relevantes
cuerpo_df = pd.read_csv(f'{DRIVE_OUT}/noticias_cuerpo.csv')
clasif_df = pd.read_csv(f'{DRIVE_OUT}/clasificacion.csv')
relevantes = clasif_df[clasif_df['llm_class'] == '1']['id']
cuerpo_df[cuerpo_df['id'].isin(relevantes)].to_csv(
    f'{DRIVE_OUT}/noticias_relevantes.csv', index=False
)
print(f'{len(relevantes)} relevantes de {len(cuerpo_df)} totales')

### Bloque 2 — después de kb

In [ ]:
merge_workers(f'{DRIVE_OUT}/extracciones.csv', 'id', DRIVE_OUT)

### Bloque 3 — después de sentiment + framer + argument + factual_backfill

In [ ]:
merge_workers(f'{DRIVE_OUT}/sentiment.csv',     'id', DRIVE_OUT)
merge_workers(f'{DRIVE_OUT}/framing.csv',       'id', DRIVE_OUT)
merge_workers(f'{DRIVE_OUT}/arguments_raw.csv', 'id', DRIVE_OUT)

# Explosión y link al KB — requiere factual_backfill completo
explode_arguments(
    raw_csv=f'{DRIVE_OUT}/arguments_raw.csv',
    output_csv=f'{DRIVE_OUT}/arguments.csv',
)

link_arguments_to_kb(
    arguments_csv=f'{DRIVE_OUT}/arguments.csv',
    extracciones_csv=f'{DRIVE_OUT}/extracciones.csv',
    menciones_raw_csv=f'{DRIVE_OUT}/menciones_raw.csv',
    clusters_csv=f'{DRIVE_OUT}/clusters.csv',
    entidades_csv=f'{DRIVE_OUT}/entidades_canonicas.csv',
    output_csv=f'{DRIVE_OUT}/arguments_linked.csv',
)